# Appendix A4 and A9

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "data" / "speeches.parquet").is_file()
)
FIG_DIR = PROJECT_ROOT / "results" / "figures"
TABLE_DIR = PROJECT_ROOT / "results" / "tables"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
df_results = pd.read_parquet(PROJECT_ROOT / "data" / "model_comparison.parquet")
gpt4 = "rating_4omini"
gpt5 = "rating_gpt55_medium"
qwen = "qwen_moral_rating"


In [ ]:
import pandas as pd
import numpy as np

from scipy.stats import kendalltau, spearmanr, pearsonr
from sklearn.metrics import cohen_kappa_score


# ------------------------------------------------------------
# Define your model rating columns
# ------------------------------------------------------------

# I assume these variables already contain the column names, e.g.:
# gpt5 = "gpt5_rating"
# gpt4 = "gpt4_rating"
# qwen = "qwen_rating"

model_pairs = [
    ("GPT-4o-mini vs GPT-5.5", gpt4, gpt5),
    ("GPT-4o-mini vs Qwen2.5", gpt4, qwen),
    ("GPT-5.5 vs Qwen2.5", gpt5, qwen),
]


# ------------------------------------------------------------
# Function to calculate comparison metrics
# ------------------------------------------------------------

def compare_rating_columns(df, col_a, col_b, label):
    data = df[[col_a, col_b]].copy()

    # Convert to numeric in case ratings are strings
    data[col_a] = pd.to_numeric(data[col_a], errors="coerce")
    data[col_b] = pd.to_numeric(data[col_b], errors="coerce")

    # Drop missing values
    data = data.dropna()

    n = len(data)

    if n < 2:
        return {
            "comparison": label,
            "n": n,
            "kendall_tau": np.nan,
            "kendall_p": np.nan,
            "spearman_rho": np.nan,
            "spearman_p": np.nan,
            "pearson_r": np.nan,
            "pearson_p": np.nan,
            "quadratic_weighted_kappa": np.nan,
            "mean_absolute_difference": np.nan,
            "exact_agreement": np.nan,
            "within_1_point": np.nan,
        }

    # Check whether either column is constant
    a_constant = data[col_a].nunique() < 2
    b_constant = data[col_b].nunique() < 2

    if a_constant or b_constant:
        kendall_corr, kendall_p = np.nan, np.nan
        spearman_corr, spearman_p = np.nan, np.nan
        pearson_corr, pearson_p = np.nan, np.nan
    else:
        kendall_corr, kendall_p = kendalltau(data[col_a], data[col_b])
        spearman_corr, spearman_p = spearmanr(data[col_a], data[col_b])
        pearson_corr, pearson_p = pearsonr(data[col_a], data[col_b])

    # Kappa needs integer categories
    ratings_a = data[col_a].round().astype(int)
    ratings_b = data[col_b].round().astype(int)

    quadratic_kappa = cohen_kappa_score(
        ratings_a,
        ratings_b,
        weights="quadratic"
    )

    abs_diff = (data[col_a] - data[col_b]).abs()

    return {
        "comparison": label,
        "n": n,
        "kendall_tau": kendall_corr,
        "kendall_p": kendall_p,
        "spearman_rho": spearman_corr,
        "spearman_p": spearman_p,
        "pearson_r": pearson_corr,
        "pearson_p": pearson_p,
        "quadratic_weighted_kappa": quadratic_kappa,
        "mean_absolute_difference": abs_diff.mean(),
        "exact_agreement": (abs_diff == 0).mean(),
        "within_1_point": (abs_diff <= 1).mean(),
    }


# ------------------------------------------------------------
# Run all pairwise comparisons
# ------------------------------------------------------------

results = []

for label, col_a, col_b in model_pairs:
    result = compare_rating_columns(
        df=df_results,
        col_a=col_a,
        col_b=col_b,
        label=label
    )
    results.append(result)

correlation_table = pd.DataFrame(results)


# ------------------------------------------------------------
# Print readable output
# ------------------------------------------------------------

for _, row in correlation_table.iterrows():
    print(row["comparison"])
    print("N:", int(row["n"]))
    print("Kendall tau:", row["kendall_tau"])
    print("Kendall p-value:", row["kendall_p"])
    print("Spearman rho:", row["spearman_rho"])
    print("Spearman p-value:", row["spearman_p"])
    print("Pearson r:", row["pearson_r"])
    print("Pearson p-value:", row["pearson_p"])
    print("Quadratic weighted kappa:", row["quadratic_weighted_kappa"])
    print("Mean absolute difference:", row["mean_absolute_difference"])
    print("Exact agreement:", row["exact_agreement"])
    print("Within 1 point:", row["within_1_point"])
    print()
    print()


# ------------------------------------------------------------
# Display dataframe
# ------------------------------------------------------------

correlation_table

In [ ]:
latex_ready = correlation_table.rename(columns={
    "comparison": "Comparison",
    "n": "N",
    "kendall_tau": "Kendall $\\tau$",
    "kendall_p": "Kendall $p$",
    "spearman_rho": "Spearman $\\rho$",
    "spearman_p": "Spearman $p$",
    "pearson_r": "Pearson $r$",
    "pearson_p": "Pearson $p$",
    "quadratic_weighted_kappa": "Weighted $\\kappa$",
    "mean_absolute_difference": "Mean abs. diff.",
    "exact_agreement": "Exact agreement",
    "within_1_point": "Within 1 point"
})

numeric_cols = latex_ready.select_dtypes(include="number").columns
latex_ready[numeric_cols] = latex_ready[numeric_cols].round(3)

latex_table = latex_ready.to_latex(
    index=False,
    caption="Correlation and agreement between model ratings",
    label="tab:model_rating_agreement",
    escape=False
)

print(latex_table)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Model columns
rating_cols = {
    "GPT-4o-mini": gpt4,
    "GPT-5.5": gpt5,
    "Qwen2.5": qwen
}

# Model-specific colors
model_colors = {
    "GPT-4o-mini": "#4C78A8",
    "GPT-5.5": "#F58518",
    "Qwen2.5": "#54A24B",
}

rating_values = [1, 2, 3, 4, 5]

# Create distribution table
dist = {}

for model_name, col in rating_cols.items():
    counts = (
        pd.to_numeric(df_results[col], errors="coerce")
        .dropna()
        .astype(int)
        .value_counts(normalize=True)
        .reindex(rating_values, fill_value=0)
    )
    dist[model_name] = counts

dist_df = pd.DataFrame(dist)

# Convert to percent
dist_percent = dist_df * 100

# Plot
x = np.arange(len(rating_values))
width = 0.25

plt.figure(figsize=(10, 6))

for i, model_name in enumerate(dist_percent.columns):
    plt.bar(
        x + (i - 1) * width,
        dist_percent[model_name],
        width,
        label=model_name,
        color=model_colors[model_name],
        edgecolor="white",
        linewidth=0.8
    )

plt.xticks(x, rating_values)
plt.xlabel("Rating")
plt.ylabel("Percent of observations")
#plt.title("Distribution of Model Ratings")
plt.legend(frameon=False)
plt.tight_layout()

plt.savefig(FIG_DIR / "model_rating_distributions_by_model_color.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
correlation_table.to_csv(TABLE_DIR / "table_a4_1.csv", index=False)
dist_percent.to_csv(TABLE_DIR / "model_rating_percentages.csv")
df_models = df_results.copy()
df_models["year"] = pd.to_datetime(df_models["date"]).dt.year


In [ ]:
# Rating columns and display names
rating_cols = {
    "GPT-4o-mini": "rating_4omini",
    "GPT-5.5": "rating_gpt55_medium",
    "Qwen2.5": "qwen_moral_rating"
}

# Make sure year and ratings are numeric
df_models["year"] = pd.to_numeric(df_models["year"], errors="coerce")

for col in rating_cols.values():
    df_models[col] = pd.to_numeric(df_models[col], errors="coerce")

# Compute yearly averages
yearly_ratings = (
    df_models
    .groupby("year")[list(rating_cols.values())]
    .mean()
    .reset_index()
    .sort_values("year")
)

# Rename for plotting
yearly_ratings = yearly_ratings.rename(columns={
    "rating_4omini": "GPT-4o-mini",
    "rating_gpt55_medium": "GPT-5.5",
    "qwen_moral_rating": "Qwen2.5"
})

# Plot
plt.figure(figsize=(14, 7))

for col in ["GPT-4o-mini", "GPT-5.5", "Qwen2.5"]:
    plt.plot(
        yearly_ratings["year"],
        yearly_ratings[col],
        marker="o",
        linewidth=2,
        label=col
    )

plt.xlabel("Year")
plt.ylabel("Average rating")
#plt.title("Yearly Average Model Ratings")
plt.ylim(1.5, 3.5)
plt.legend(frameon=False)
plt.tight_layout()

plt.savefig(FIG_DIR / "yearly_average_model_ratings.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
yearly_ratings.to_csv(TABLE_DIR / "model_yearly_means.csv", index=False)

In [ ]:
def ordinal_alpha(ratings):
    """Krippendorff ordinal alpha; rows are speeches, columns are raters.

    Coincidences weight each unit by 1/(available raters - 1). Ordinal distances
    use pooled category frequencies between the two categories, with half of
    each endpoint frequency. Units with fewer than two ratings are excluded.
    """
    coincidence = np.zeros((5, 5), dtype=float)
    for row in np.asarray(ratings, dtype=float):
        values = row[np.isfinite(row)].astype(int)
        if len(values) >= 2:
            counts = np.bincount(values, minlength=6)[1:6]
            coincidence += (np.outer(counts, counts) - np.diag(counts)) / (len(values) - 1)
    marginal = coincidence.sum(axis=0)
    distance = np.zeros((5, 5))
    for i in range(5):
        for j in range(i + 1, 5):
            distance[i, j] = distance[j, i] = (
                marginal[i:j + 1].sum() - (marginal[i] + marginal[j]) / 2
            ) ** 2
    expected = (np.outer(marginal, marginal) * distance).sum()
    return float(1 - (marginal.sum() - 1) * (coincidence * distance).sum() / expected) if expected else float("nan")

In [ ]:
human = pd.read_csv(PROJECT_ROOT / "data" / "human_validation.csv")
human_columns = ["expert_1_rating", "expert_2_rating", "llm_rating"]
human[human_columns] = human[human_columns].apply(pd.to_numeric, errors="coerce")
comparisons = [
    ("Expert 1 vs Expert 2", "expert_1_rating", "expert_2_rating"),
    ("Expert 1 vs LLM", "expert_1_rating", "llm_rating"),
    ("Expert 2 vs LLM", "expert_2_rating", "llm_rating"),
]
rows = []
for label, left, right in comparisons:
    row = compare_rating_columns(human, left, right, label)
    row["ordinal_alpha"] = ordinal_alpha(human[[left, right]].dropna())
    rows.append(row)
human_agreement = pd.DataFrame(rows)
human_agreement.to_csv(TABLE_DIR / "table_a9_1.csv", index=False)
print("Three-coder ordinal alpha:", ordinal_alpha(human[human_columns]))
human_agreement
